# DeepSeek OCR — Inferencia V5

Notebook de pruebas para el adapter `Lacax/deepseek_ocr_lora_v5` (entrenado con dataset golden).

## Diferencias respecto al notebook V4

1. **Adapter HF**: `Lacax/deepseek_ocr_lora_v5` (no sobrescribe V4).
2. **Prompt idéntico al training** (`INSTRUCTION` constante): incluye `fecha_original` y `cantidad: number|null` (V5-H/I).
3. **Sin `FastVisionModel.for_inference`** en celda 4 (innecesario y problemático con deepseek_vl_v2).
4. **Celda Gradio** al final para subir un ticket y ver la salida en navegador.

Compatible con Google Colab (T4 free / A100 pro). Tiempo aprox: 30–60 s por ticket en T4.

### Celda 0 — Setup (ejecutar UNA vez, luego Restart)

In [ ]:
# [CELDA 0] SETUP
# 1. Instala transformers==4.56.2 (version de entrenamiento)
# 2. Restaura modeling_deepseekocr2.py original desde HuggingFace
# 3. Limpia cache HF
import subprocess, shutil, os

HF_TOKEN = "TU_HF_TOKEN"  # <-- pon tu token aqui

subprocess.run(["pip", "install",
    "transformers==4.56.2",
    "peft", "accelerate", "pillow",
    "huggingface_hub", "addict", "safetensors",
    "opencv-python-headless", "hf_transfer",
    "-q"], check=True)
print("transformers==4.56.2 instalado")

from huggingface_hub import hf_hub_download
original = hf_hub_download(
    "unsloth/DeepSeek-OCR-2",
    "modeling_deepseekocr2.py",
    token=HF_TOKEN,
    force_download=True,
)
dest = "./deepseek_ocr2/modeling_deepseekocr2.py"
if os.path.exists("./deepseek_ocr2"):
    shutil.copy(original, dest)
    print(f"modeling_deepseekocr2.py restaurado -> {dest}")
else:
    print("deepseek_ocr2/ no existe aun (se creara en celda 2)")

cache_path = os.path.expanduser("~/.cache/huggingface/modules/")
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print("Cache HF limpiada")

print("=" * 50)
print("LISTO. Kernel -> Restart, luego ejecuta celdas 1, 2, 3...")
print("=" * 50)

### Celda 1 — Inspeccionar adapter V5

In [ ]:
# [CELDA 1] Inspeccion de pesos del adaptador LoRA V5
from safetensors import safe_open
from huggingface_hub import snapshot_download
import torch, os

HF_TOKEN = "TU_HF_TOKEN"  # <-- pon tu token aqui
ADAPTER_REPO = "Lacax/deepseek_ocr_lora_v5"  # V5

if not os.path.exists("./adapter_v5/adapter_model.safetensors"):
    snapshot_download(ADAPTER_REPO, local_dir="./adapter_v5", token=HF_TOKEN)

results = {"critical": [], "lora_B_zero": [], "lora_A_dead": [], "other_warnings": []}
with safe_open("./adapter_v5/adapter_model.safetensors", framework="pt") as f:
    for key in f.keys():
        tensor = f.get_tensor(key)
        if torch.isnan(tensor).any():
            results["critical"].append(f"NaN en {key}")
        if torch.isinf(tensor).any():
            results["critical"].append(f"Inf en {key}")
        if (tensor == 0).float().mean() > 0.9:
            (results["lora_B_zero"] if "lora_B" in key else results["lora_A_dead"]).append(key)
        if tensor.float().std() < 1e-7 and "lora_B" not in key:
            results["other_warnings"].append(f"Varianza nula: {key}")

print("CRITICOS:", results["critical"])
print("lora_B a cero (NORMAL si no entreno):", len(results["lora_B_zero"]), "capas")
print("lora_A MUERTA (PROBLEMA):", results["lora_A_dead"])
print("OTRAS advertencias:", results["other_warnings"])

# Tras un entrenamiento exitoso, lora_B NO debe estar todo a cero (significaria que el adapter no aprendio nada).
# Para V5 tras 6 epocas con eval_loss=0.13, esperamos lora_B con valores no triviales.

### Celda 2 — Cargar modelo base + adapter V5

In [ ]:
# [CELDA 2] Carga del modelo base + adaptador LoRA V5
import torch, os
from transformers import AutoModel, AutoTokenizer
from peft import PeftModel
from huggingface_hub import snapshot_download

HF_TOKEN = "TU_HF_TOKEN"  # <-- pon tu token aqui
ADAPTER_REPO = "Lacax/deepseek_ocr_lora_v5"  # V5

if not os.path.exists("./deepseek_ocr2/config.json"):
    snapshot_download("unsloth/DeepSeek-OCR-2", local_dir="./deepseek_ocr2", token=HF_TOKEN)
else:
    print("deepseek_ocr2 ya descargado")

model = AutoModel.from_pretrained(
    "./deepseek_ocr2",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
)
tokenizer = AutoTokenizer.from_pretrained("./deepseek_ocr2", trust_remote_code=True)

model = PeftModel.from_pretrained(model, ADAPTER_REPO, token=HF_TOKEN)
model.eval()
print(f"OK V5 — dtype={model.dtype}, device={next(model.parameters()).device}")
print(f"Adapter: {ADAPTER_REPO}")

### Celda 3 — Utilidades compartidas (preprocess + collator + prompt V5)

Esta celda define `preprocess_ticket`, `DeepSeekOCR2DataCollator`, `INSTRUCTION` (idéntico al training V5) y `run_inference()`. Las celdas 4-6 lo reutilizan.

In [ ]:
# [CELDA 3] Utilidades compartidas
import torch, math, json, os, io
import cv2
import numpy as np
from dataclasses import dataclass
from typing import Any
from PIL import Image, ImageOps
from torch.nn.utils.rnn import pad_sequence
from deepseek_ocr2.modeling_deepseekocr2 import (
    text_encode,
    BasicImageTransform,
    dynamic_preprocess,
)

# V5-H: PROMPT IDENTICO al INSTRUCTION usado en training (Celda E del notebook V5).
# Cambiar aqui implica re-entrenar. Schema V5-I: incluye fecha_original y cantidad: number|null.
INSTRUCTION = """<image>
Extract the receipt info as a JSON object with this exact structure:
{\"comercio\": \"string\", \"cif\": \"string\", \"fecha\": \"YYYY-MM-DD\", \"fecha_original\": \"string\", \"total\": \"number\", \"items\": [{\"cantidad\": \"number|null\", \"descripcion\": \"string\", \"precio\": \"number\"}]}
NO other text. ONLY valid JSON.
"""

# H1.5: deskew + crop margenes blancos
def preprocess_ticket(img: Image.Image) -> Image.Image:
    arr = np.array(img.convert("L"))
    _, thresh = cv2.threshold(arr, 200, 255, cv2.THRESH_BINARY_INV)
    coords = np.column_stack(np.where(thresh > 0))
    if len(coords) >= 50:
        angle = cv2.minAreaRect(coords)[-1]
        if angle < -45:
            angle = 90 + angle
        if abs(angle) > 0.5:
            h, w = arr.shape
            M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
            img = Image.fromarray(
                cv2.warpAffine(np.array(img), M, (w, h),
                               flags=cv2.INTER_CUBIC,
                               borderMode=cv2.BORDER_REPLICATE)
            )
    arr2 = np.array(img.convert("L"))
    _, thresh2 = cv2.threshold(arr2, 200, 255, cv2.THRESH_BINARY_INV)
    rows = np.any(thresh2 > 0, axis=1)
    cols = np.any(thresh2 > 0, axis=0)
    if rows.any() and cols.any():
        rmin, rmax = np.where(rows)[0][[0, -1]]
        cmin, cmax = np.where(cols)[0][[0, -1]]
        pad = 10
        img = img.crop((
            max(0, cmin - pad), max(0, rmin - pad),
            min(img.width, cmax + pad), min(img.height, rmax + pad)
        ))
    return img


class DeepSeekOCR2DataCollator:
    def __init__(self, tokenizer, model, image_size=768, base_size=1024,
                 crop_mode=True, train_on_responses_only=False):
        self.tokenizer = tokenizer
        self.model = model
        self.image_size = image_size
        self.base_size = base_size
        self.crop_mode = crop_mode
        self.image_token_id = 128815
        self.dtype = model.dtype
        self.train_on_responses_only = train_on_responses_only
        self.image_transform = BasicImageTransform(
            mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5), normalize=True
        )
        self.patch_size = 16
        self.downsample_ratio = 4
        self.bos_id = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else 0

    def deserialize_image(self, image_data):
        if isinstance(image_data, Image.Image):
            return ImageOps.exif_transpose(image_data.convert("RGB"))
        elif isinstance(image_data, dict) and 'bytes' in image_data:
            return ImageOps.exif_transpose(Image.open(io.BytesIO(image_data['bytes'])).convert("RGB"))
        elif isinstance(image_data, str) and os.path.exists(image_data):
            return ImageOps.exif_transpose(Image.open(image_data).convert("RGB"))
        raise ValueError(f"Unsupported image format: {type(image_data)}")

    def process_image(self, image):
        images_list, images_crop_list, images_spatial_crop = [], [], []
        if self.crop_mode:
            images_crop_raw, crop_ratio = dynamic_preprocess(
                image, min_num=1, max_num=6,
                image_size=self.image_size, use_thumbnail=False
            )
            global_view = ImageOps.pad(
                image, (self.base_size, self.base_size),
                color=tuple(int(x * 255) for x in self.image_transform.mean)
            )
            images_list.append(self.image_transform(global_view).to(self.dtype))
            width_crop_num, height_crop_num = crop_ratio
            images_spatial_crop.append([width_crop_num, height_crop_num])
            if width_crop_num > 1 or height_crop_num > 1:
                for crop_img in images_crop_raw:
                    images_crop_list.append(self.image_transform(crop_img).to(self.dtype))
            num_queries_base = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)
            tokenized_image = ([self.image_token_id] * num_queries_base) * num_queries_base
            tokenized_image += [self.image_token_id]
        return images_list, images_crop_list, images_spatial_crop, tokenized_image, crop_ratio

    def process_single_sample(self, messages):
        images = []
        for message in messages:
            if "images" in message and message["images"]:
                for img_data in message["images"]:
                    if img_data is not None:
                        images.append(self.deserialize_image(img_data))
        if not images:
            raise ValueError("No images found in sample.")
        tokenized_str, images_seq_mask = [], []
        images_list, images_crop_list, images_spatial_crop = [], [], []
        prompt_token_count = -1
        assistant_started = False
        image_idx = 0
        tokenized_str.append(self.bos_id)
        images_seq_mask.append(False)
        for message in messages:
            role = message["role"]
            content = message["content"]
            if role == "<|Assistant|>":
                if not assistant_started:
                    prompt_token_count = len(tokenized_str)
                    assistant_started = True
                if content.strip():
                    content = f"{content.strip()} {self.tokenizer.eos_token}"
            text_splits = content.split('<image>')
            for i, text_sep in enumerate(text_splits):
                tokenized_sep = text_encode(self.tokenizer, text_sep, bos=False, eos=False)
                tokenized_str.extend(tokenized_sep)
                images_seq_mask.extend([False] * len(tokenized_sep))
                if i < len(text_splits) - 1:
                    if image_idx >= len(images):
                        raise ValueError("Data mismatch: more <image> tokens than images.")
                    img_list, crop_list, spatial_crop, tok_img, _ = self.process_image(images[image_idx])
                    images_list.extend(img_list)
                    images_crop_list.extend(crop_list)
                    images_spatial_crop.extend(spatial_crop)
                    tokenized_str.extend(tok_img)
                    images_seq_mask.extend([True] * len(tok_img))
                    image_idx += 1
        if not assistant_started:
            prompt_token_count = len(tokenized_str)
        images_ori = torch.stack(images_list, dim=0)
        images_spatial_crop_tensor = torch.tensor(images_spatial_crop, dtype=torch.long)
        images_crop = (torch.stack(images_crop_list, dim=0) if images_crop_list
                       else torch.zeros((1, 3, self.base_size, self.base_size), dtype=self.dtype))
        return {
            "input_ids": torch.tensor(tokenized_str, dtype=torch.long),
            "images_seq_mask": torch.tensor(images_seq_mask, dtype=torch.bool),
            "images_ori": images_ori,
            "images_crop": images_crop,
            "images_spatial_crop": images_spatial_crop_tensor,
            "prompt_token_count": prompt_token_count,
        }

    def __call__(self, features):
        batch_data = []
        for feature in features:
            try:
                batch_data.append(self.process_single_sample(feature['messages']))
            except Exception as e:
                print(f"Error procesando muestra: {e}")
        if not batch_data:
            raise ValueError("No valid samples in batch")
        input_ids_list = [item['input_ids'] for item in batch_data]
        images_seq_mask_list = [item['images_seq_mask'] for item in batch_data]
        prompt_token_counts = [item['prompt_token_count'] for item in batch_data]
        input_ids = pad_sequence(input_ids_list, batch_first=True, padding_value=self.tokenizer.pad_token_id)
        images_seq_mask = pad_sequence(images_seq_mask_list, batch_first=True, padding_value=False)
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        labels[images_seq_mask] = -100
        if self.train_on_responses_only:
            for idx, prompt_count in enumerate(prompt_token_counts):
                if prompt_count > 0:
                    labels[idx, :prompt_count] = -100
        attention_mask = (input_ids != self.tokenizer.pad_token_id).long()
        images_batch = [(item['images_crop'], item['images_ori']) for item in batch_data]
        images_spatial_crop = torch.cat([item['images_spatial_crop'] for item in batch_data], dim=0)
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "images": images_batch,
            "images_seq_mask": images_seq_mask,
            "images_spatial_crop": images_spatial_crop,
        }


inference_collator = DeepSeekOCR2DataCollator(
    tokenizer=tokenizer,
    model=model,
    image_size=768,
    base_size=1024,
    crop_mode=True,
    train_on_responses_only=False,
)


def run_inference(image: Image.Image):
    image = preprocess_ticket(image)
    sample = {
        "messages": [
            {"role": "<|User|>", "content": INSTRUCTION, "images": [image]},
            {"role": "<|Assistant|>", "content": ""},
        ]
    }
    batch = inference_collator([sample])
    input_ids           = batch["input_ids"].to(model.device)
    attention_mask      = batch["attention_mask"].to(model.device)
    images              = [(c.to(model.device, model.dtype), o.to(model.device, model.dtype)) for c, o in batch["images"]]
    images_seq_mask     = batch["images_seq_mask"].to(model.device)
    images_spatial_crop = batch["images_spatial_crop"].to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids, attention_mask=attention_mask,
            images=images, images_seq_mask=images_seq_mask,
            images_spatial_crop=images_spatial_crop,
            max_new_tokens=4096, do_sample=False, repetition_penalty=1.0,
        )

    response = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
    try:
        return response, True, json.loads(response)
    except json.JSONDecodeError:
        return response, False, None

print("Utilidades cargadas. INSTRUCTION = prompt V5 (con fecha_original y cantidad number|null).")

### Celda 4 — Test rápido de un único ticket

Sube una imagen al directorio de trabajo y pon su nombre en `IMG`. Salida: JSON parseado + raw output.

In [ ]:
# [CELDA 4] Test rapido — un solo ticket
from PIL import Image
import json

IMG = "recibo_almeria_079.jpg"  # <-- cambia por tu ticket

image = Image.open(IMG).convert("RGB")
raw, valid, parsed = run_inference(image)

print("="*60)
print(f"Imagen: {IMG}")
print("="*60)
print("Raw output:")
print(raw)
print()
if valid:
    print("-> JSON VALIDO")
    print(json.dumps(parsed, ensure_ascii=False, indent=2))
else:
    print("-> JSON INVALIDO")

### Celda 5 — Tests A-E (protocolo PLAN)

Ejecuta el protocolo completo: 5 tickets para sanidad (A), out-of-domain (B), consistencia (C), campo faltante (D), comercio nuevo (E).

Imágenes esperadas en el directorio:
- `recibo_almeria_079/110/111/112/114.jpg` (Test A)
- `vikings.jpg` (Test B — imagen sin texto)
- `recibo_almeria_079.jpg` (Tests C y D)
- `recibo_diferente.jpg` (Test E — comercio no visto)

In [ ]:
# [CELDA 5] Tests A-E
from PIL import Image
import os, json

# ---- TEST A — sanidad con 5 tickets ----------------------------------------
print("="*60)
print("TEST A — Sanidad (5 tickets)")
print("="*60)
IMAGENES_A = [
    "recibo_almeria_079.jpg",
    "recibo_almeria_110.jpg",
    "recibo_almeria_111.jpg",
    "recibo_almeria_112.jpg",
    "recibo_almeria_114.jpg",
]
resultados_a = []
for img_path in IMAGENES_A:
    if not os.path.exists(img_path):
        print(f"  SKIP: {img_path} no encontrada")
        continue
    image = Image.open(img_path).convert("RGB")
    raw, valid, parsed = run_inference(image)
    status = "PASS" if valid else "FAIL"
    comercio = parsed.get("comercio", "?") if parsed else "INVALID"
    total = parsed.get("total", "?") if parsed else "INVALID"
    print(f"  {img_path}: {status} | comercio={comercio} | total={total}")
    resultados_a.append((img_path, valid, parsed, raw))

# ---- TEST B — out-of-domain ------------------------------------------------
print("\n" + "="*60)
print("TEST B — Imagen fuera de dominio")
print("="*60)
IMG_B = "vikings.jpg"
if not os.path.exists(IMG_B):
    print(f"SKIP: {IMG_B} no encontrada")
else:
    raw_b, valid_b, parsed_b = run_inference(Image.open(IMG_B).convert("RGB"))
    print("Raw:", raw_b[:300])
    if valid_b:
        nulos = all(parsed_b.get(k) in (None, "", "null", "N/A") for k in ["comercio", "cif", "total"])
        print("-> PASS" if nulos else "-> FAIL (inventa datos)")
    else:
        print("-> JSON invalido — revisar manualmente si rechaza")

# ---- TEST C — consistencia x5 ----------------------------------------------
print("\n" + "="*60)
print("TEST C — Consistencia (misma imagen x5)")
print("="*60)
IMG_C = "recibo_almeria_079.jpg"
if not os.path.exists(IMG_C):
    print(f"SKIP: {IMG_C} no encontrada")
else:
    image_c = Image.open(IMG_C).convert("RGB")
    runs = []
    for i in range(5):
        raw, valid, parsed = run_inference(image_c)
        runs.append((valid, parsed))
        com = parsed.get("comercio", "?") if parsed else "INVALID"
        tot = parsed.get("total", "?") if parsed else "INVALID"
        print(f"  Run {i+1}: {'OK' if valid else 'FAIL'} | comercio={com} | total={tot}")
    if all(v for v, _ in runs):
        totales = set(str(p.get("total")) for _, p in runs)
        comercios = set(p.get("comercio", "") for _, p in runs)
        print(f"  Totales unicos: {totales}")
        print(f"  Comercios unicos: {comercios}")
        print("-> PASS" if (len(totales)==1 and len(comercios)==1) else "-> WARN (variaciones)")
    else:
        print("-> FAIL")

# ---- TEST D — campo faltante (recorte 60%) ---------------------------------
print("\n" + "="*60)
print("TEST D — Ticket con zona de total recortada")
print("="*60)
if not os.path.exists(IMG_C):
    print("SKIP")
else:
    img_full = Image.open(IMG_C).convert("RGB")
    w, h = img_full.size
    img_cropped = img_full.crop((0, 0, w, int(h * 0.60)))
    raw_d, valid_d, parsed_d = run_inference(img_cropped)
    print("Raw:", raw_d[:300])
    if valid_d:
        total_v = parsed_d.get("total")
        ausente = total_v in (None, 0, "", "null")
        print(f"  total extraido: {total_v}")
        print("-> PASS" if ausente else "-> WARN (verificar si inventa o estaba en otro lugar)")
    else:
        print("-> FAIL")

# ---- TEST E — comercio no visto -------------------------------------------
print("\n" + "="*60)
print("TEST E — Comercio no visto en entrenamiento")
print("="*60)
IMG_E = "recibo_diferente.jpg"
if not os.path.exists(IMG_E):
    print(f"SKIP: {IMG_E} no encontrada (sube un ticket de Carrefour/Lidl/Aldi/etc.)")
else:
    raw_e, valid_e, parsed_e = run_inference(Image.open(IMG_E).convert("RGB"))
    print("Raw:", raw_e[:300])
    COMERCIOS_TRAIN = {"mercadona", "grupo dia", "super alcarro", "dia"}
    if valid_e:
        com = (parsed_e.get("comercio") or "").lower().strip()
        overfitting = any(c in com for c in COMERCIOS_TRAIN)
        print(f"  comercio extraido: {parsed_e.get('comercio')}")
        print("-> FAIL (overfit)" if overfitting else "-> PASS")
    else:
        print("-> FAIL (JSON invalido)")

### Celda 6 — Demo Gradio

Lanza una interfaz web para subir tickets desde el navegador. En Colab usa `share=True` para obtener una URL pública (gradio.live) durante 72h.

Reutiliza `model`, `tokenizer`, `inference_collator` y `INSTRUCTION` de las celdas anteriores.

In [ ]:
# [CELDA 6] Gradio demo
import subprocess
subprocess.run(["pip", "install", "gradio", "-q"], check=True)

import gradio as gr
from PIL import Image
import json

# Fixes Unicode (a veces el modelo emite comas/comillas tipograficas chinas)
UNICODE_FIX = {
    "\uFF0C": ",", "\uFF1A": ":",
    "\u201C": '"', "\u201D": '"',
}
def fix_unicode(text):
    for bad, good in UNICODE_FIX.items():
        text = text.replace(bad, good)
    return text

def predict(image):
    try:
        if image is None:
            return "Sin imagen", "{}"
        if not isinstance(image, Image.Image):
            image = Image.fromarray(image).convert("RGB")
        raw, valid, parsed = run_inference(image)
        if valid:
            return "OK JSON valido", json.dumps(parsed, ensure_ascii=False, indent=2)
        # intento de recuperacion con fix unicode
        fixed = fix_unicode(raw.strip())
        try:
            parsed = json.loads(fixed)
            return "OK JSON valido (unicode fixed)", json.dumps(parsed, ensure_ascii=False, indent=2)
        except json.JSONDecodeError:
            return "JSON invalido", raw.strip()
    except Exception as e:
        import traceback
        return f"EXCEPCION: {e}", traceback.format_exc()

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(label="Sube un ticket"),
    outputs=[
        gr.Textbox(label="Estado"),
        gr.Code(label="JSON extraido", language="json"),
    ],
    title="DeepSeek OCR — Fine-tuned V5",
    description="Adapter Lacax/deepseek_ocr_lora_v5 (eval_loss 0.1274, 6 epocas, dataset golden 816 tickets).",
    flagging_mode="never",
)
demo.launch(share=True)